<img src="https://datasciencedegree.wisconsin.edu/wp-content/themes/data-gulp/images/logo.svg" width="300">

 # <center>DS 710 - Lesson 8</center>
<center>Text Analysis with Python and NLTK</center>


This notebook will work through some of the basics of the nltk package, including

1. The NLTK corpus
1. Tokenization
2. Removal of Stopwords
3. Sentiment Analysis

---

References for the material in this notebook include the NLTK documentation
* https://www.nltk.org/
* https://www.nltk.org/book/
* https://www.nltk.org/howto/sentiment.html

---

## 0. Import NLTK and download some Corpora

Start by importing the nltk package.

In [ ]:
import nltk
import pandas as pd #pandas will be used later in the workbook

We will also download certain useful corpora. You only ever have to do this download process once per instance of Python. 

For the notebook, let's get `"stopwords"`, `"gutenberg"` and `"webtexts"`.

* `"stopwords"` includes basic words that are generally filler words.
* `"gutenberg"` includes some texts from Project Gutenberg.
* `"webtexts"` includes a variety of online texts, which we'll explore.



In [ ]:
# we only have to do this once per corpus (well per instance of python)

nltk.download("stopwords")

nltk.download("gutenberg")

nltk.download("webtext")

nltk.download("punkt")
nltk.download('vader_lexicon')

---

## 1. Stopwords and other Libraries

### Stopwords first

To be more clear about what these are, let's just explore them quickly.


In [ ]:
print(nltk.corpus.stopwords.words("english"))

🧠 What words in here did you expect, what did you not expect? 

🎯 Explore the stopwords in another language.

### Gutenberg

Now let's see what is available in the `"gutenberg"` corpus.  

You will notice, there's a few Jane Austen files, Alice in Wonderland, some Shakespeare, and a few more.

In [ ]:
nltk.corpus.gutenberg.fileids()

🎯 Use tab complete after `nltk.corpus.gutenberg` to see a few options for usage.

For example, try `nltk.corpus.gutenberg.words(fileid)`

In [ ]:
nltk.corpus.gutenberg.words('melville-moby_dick.txt')

### Webtext

Webtext is a collection of some text files gathered from online sources. Let's see what's there.

In [ ]:
nltk.corpus.webtext.fileids()

🎯 We're going to get into `"wine.txt"` in the next section, but for now, take a look at some of these.

---

## 2. Preparing Text for Sentiment Analysis

Sentiment analysis is best completed on tokenized text.  So let's walk through some standard tools in `nltk` to do this preparation, recalling what we did in Lesson 3b and 4.


For this practice, we're going to work with `"wine.txt"` from `nltk.corpus.webtext`.  So let's save that raw text to a variable.

The file `"wine.txt"` contains some wine reviews slurped from some online forum.



In [ ]:
raw_wine = nltk.corpus.webtext.raw("wine.txt")

In [ ]:
raw_wine

It looks like the reviews are separated by a single `'\n'`, and the review scores are given by the number of `"*"` at the end of the lines.  So let's translate this information to a dataframe with columns `"Review Text"` and `"Rating"`.

🎯 Turn the `review_tuples` variable into a dataframe `wine_df`.

In [ ]:
review_tuples = []

for review in raw_wine.split("\n"):
    if review != "":
        num_stars = 0
        while "*" in review[-2:]:
            if review[-3:] == "(*)":
                num_stars += 0.5
                review = review[:-3]
            else:
                num_stars += 1
                review = review[:-1]
        review_tuples.append((review,num_stars))

review_tuples[:10]

In [ ]:
wine_df = pd.DataFrame(review_tuples,columns=["Review Text","Rating"])

So now let's talk about processing.  There are a few general steps that happen when processing text for sentiment analysis:

1. Tokenization of words
2. Removal of stopwords

🎯 `apply` the `prepare` function here to the `"Review Text"` column in `wine_df`, creating a new column called `"Prepared Text"`.

In [ ]:
def prepare(review_text):
    # Tokenize
    tokenized = nltk.tokenize.word_tokenize(review_text)

    # Remove Stopwords
    stopwords = nltk.corpus.stopwords.words("english")
    tokenized = [t for t in tokenized if t not in stopwords]

    return " ".join(tokenized)

In [ ]:
wine_df["Prepared Text"] = wine_df["Review Text"].apply(prepare)
wine_df["Prepared Text"]

---

## 3. Sentiment Analysis of text in a Dataframe

🎯 Let's import the sentiment intensity analyzer.

In [ ]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer

🎯 Run the following cell to see the dictionary created by the analyzer when run on some text.

In [ ]:

def analyzer(review):
    sia = SentimentIntensityAnalyzer()
    scores = sia.polarity_scores(review)
    return scores

analyzer(wine_df["Prepared Text"][0])

Some interesting notes about these scores:
1. the sum of the `'neg'`, `'neu'`, and `'pos'` values is 1. They each give some ratio of the negative, neutral, or positive sentiment of the text.
2. The `'compound'` score can be between -1 and 1, and it is some weighted average of the other three scores.  Negative compound values indicate negative sentiment and positive values indicate positive sentiment.  
3. The scores here have been trained using machine learning.  Developers input phrases and sentences along with their sentiment values.  Then some linear algebra and other mathematics come together to take the input data and generate probabilities on new data.

🎯 Use `pd.concat` to add columns named `'neg'`, `'neu'`, `'pos'`, and `'compound'` to `wine_df`.

⚠ You probably want an `axis=1`.

In [ ]:
scores = []

for review in wine_df["Prepared Text"]:
    scores.append(analyzer(review))

wine_df = pd.concat

---

## 4. Compound vs Ratings Plots

Let's compare the compound sentiment scores with the ratings given by the reviewers.

🎯 Explore a few more plot types to see if you can find some correlation between Rating and compound scores.

🎯 Try dropping the 0 scores and see if that changes the correlation.

In [ ]:
wine_df.plot.scatter(x="Rating",y="compound")

---

## When you're done

🧠 After completing the guided practice above, log onto Canvas to complete Quiz 8. You may wish to keep this notebook open while you work.

Credits:

* Prof. Mckenzie West: Spring 2024